# Notebook 06 — Popularity Regression

Notebook chạy lại Linear Regression, Random Forest và XGBoost trên ba contract: Baseline With-Time, Engineered With-Time, Engineered No-Time. Winner được chọn từ **tất cả eligible experiments**, không ép engineered model thắng.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for candidate in Path.cwd().resolve().parents:
        if (candidate / "src").exists() and (candidate / "5.DATA").exists():
            ROOT = candidate
            break
sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

from sklearn.metrics import mean_absolute_error, mean_squared_error

from src.features import (
    RAW_INPUT_FEATURES, SELECTED_ENGINEERED_FEATURES, TARGET,
    TEST_START_YEAR, FeatureBuilder, get_model_features,
)
from src.modeling import (
    MODEL_NAMES, build_model_pipeline, grouped_feature_importance,
    regression_metric_variants, transformed_feature_importance,
)

DATA_PATH = ROOT / "5.DATA" / "processed" / "ml_ready_dataset.parquet"
ENGINEERED_PATH = ROOT / "5.DATA" / "processed" / "features_engineered.parquet"
EVAL_DIR = ROOT / "4.MODELS" / "4.2.evaluation"
MODEL_DIR = ROOT / "4.MODELS" / "hitradar_popularity"
for directory in (EVAL_DIR, MODEL_DIR): directory.mkdir(parents=True, exist_ok=True)
assert DATA_PATH.exists() and ENGINEERED_PATH.exists(), "Run Notebook 05 first."

Project root: D:\Hitradar\hitradar-main


## 1. Load output Notebook 05 và parity test

Parity test chứng minh saved dataframe và shared builder tạo cùng giá trị. Numeric dùng tolerance chặt; categorical so khớp chính xác. Kết quả được save, không chỉ in ra.

In [2]:
raw = pd.read_parquet(DATA_PATH)
saved = pd.read_parquet(ENGINEERED_PATH)
assert len(raw) == len(saved)
train_mask = raw["release_year"] < TEST_START_YEAR
test_mask = ~train_mask
X_train, X_test = raw.loc[train_mask, RAW_INPUT_FEATURES], raw.loc[test_mask, RAW_INPUT_FEATURES]
y_train, y_test = raw.loc[train_mask, TARGET], raw.loc[test_mask, TARGET]

parity_builder = FeatureBuilder(include_engineered=True).fit(X_train)
parity_index = raw.loc[test_mask].sample(min(10_000, int(test_mask.sum())), random_state=1512).index
rebuilt = parity_builder.transform(raw.loc[parity_index, RAW_INPUT_FEATURES]).reset_index(drop=True)
expected = saved.loc[parity_index, rebuilt.columns].reset_index(drop=True)
numeric = rebuilt.select_dtypes(include=np.number).columns.tolist()
categorical = [c for c in rebuilt.columns if c not in numeric]
numeric_ok = bool(np.allclose(rebuilt[numeric], expected[numeric], rtol=1e-9, atol=1e-10, equal_nan=True))
categorical_ok = all(rebuilt[c].astype("string").equals(expected[c].astype("string")) for c in categorical)
parity_result = {"rows_checked":len(rebuilt), "numeric_allclose":numeric_ok, "categorical_exact":categorical_ok, "status":"PASS" if numeric_ok and categorical_ok else "FAIL"}
assert parity_result["status"] == "PASS", parity_result
(EVAL_DIR / "feature_builder_saved_parity.json").write_text(json.dumps(parity_result, indent=2), encoding="utf-8")
print(parity_result)

{'rows_checked': 10000, 'numeric_allclose': True, 'categorical_exact': True, 'status': 'PASS'}


## 2. Train all eligible experiments

Category features được impute + one-hot encode trong pipeline. Scaler/encoder/learned features chỉ fit bằng train. Metrics báo cả raw predictions và production-clipped [0,100].

In [3]:
experiments = [
    {"Experiment":"Baseline With-Time", "include_engineered":False, "include_time":True},
    {"Experiment":"Engineered With-Time", "include_engineered":True, "include_time":True},
    {"Experiment":"Engineered No-Time", "include_engineered":True, "include_time":False},
]
rows, fitted, predictions = [], {}, {}
for experiment in experiments:
    declared = get_model_features(include_engineered=experiment["include_engineered"], include_time=experiment["include_time"])
    assert declared and all(f in parity_builder.transform(X_train.head(3)).columns for f in declared)
    for model_name in MODEL_NAMES:
        key = (experiment["Experiment"], model_name)
        print("Fitting", key)
        pipeline = build_model_pipeline(model_name, include_engineered=experiment["include_engineered"], include_time=experiment["include_time"])
        pipeline.fit(X_train, y_train)
        pred_raw = pipeline.predict(X_test)
        fitted[key] = pipeline
        predictions[key] = pred_raw
        for variant, metric in regression_metric_variants(y_test, pred_raw).items():
            rows.append({"Experiment":key[0], "Model":model_name, "Prediction Variant":variant, "Feature Count":len(declared), **metric})
metrics_table = pd.DataFrame(rows).sort_values(["Prediction Variant", "RMSE"]).reset_index(drop=True)
display(metrics_table)

Fitting ('Baseline With-Time', 'Linear Regression')


Fitting ('Baseline With-Time', 'Random Forest')


Fitting ('Baseline With-Time', 'XGBoost')


Fitting ('Engineered With-Time', 'Linear Regression')


Fitting ('Engineered With-Time', 'Random Forest')


Fitting ('Engineered With-Time', 'XGBoost')


Fitting ('Engineered No-Time', 'Linear Regression')


Fitting ('Engineered No-Time', 'Random Forest')


Fitting ('Engineered No-Time', 'XGBoost')


,Experiment,Model,Prediction Variant,Feature Count,MAE,RMSE,R2
0,Baseline With-Time,XGBoost,"Clipped [0,100]",18,16.302257,20.576374,0.260362
1,Engineered With-Time,XGBoost,"Clipped [0,100]",32,16.201599,20.594952,0.259026
2,Engineered With-Time,Random Forest,"Clipped [0,100]",32,16.369137,20.683873,0.252613
3,Baseline With-Time,Random Forest,"Clipped [0,100]",18,16.520659,20.707796,0.250884
4,Engineered With-Time,Linear Regression,"Clipped [0,100]",32,18.335150,22.700191,0.099797
5,Baseline With-Time,Linear Regression,"Clipped [0,100]",18,18.633607,22.913068,0.082834
6,Engineered No-Time,XGBoost,"Clipped [0,100]",26,19.563436,22.963482,0.078793
7,Engineered No-Time,Random Forest,"Clipped [0,100]",26,19.812843,23.179310,0.061396
8,Engineered No-Time,Linear Regression,"Clipped [0,100]",26,21.672246,24.733624,-0.068703
9,Baseline With-Time,XGBoost,Raw,18,16.313208,20.579572,0.260132


## 3. Winner selection toàn cục và bias comparisons

Selection dùng clipped RMSE vì API trả popularity trong miền hợp lệ. Raw metrics vẫn được lưu để minh bạch ảnh hưởng clipping.

In [4]:
eligible = metrics_table.query("`Prediction Variant` == 'Clipped [0,100]'").copy()
winner_row = eligible.sort_values(["RMSE", "MAE", "Model", "Experiment"]).iloc[0]
winner_key = (winner_row["Experiment"], winner_row["Model"])
final_pipeline = fitted[winner_key]
winner_experiment = next(e for e in experiments if e["Experiment"] == winner_key[0])
print("FINAL WINNER:", winner_key)
display(winner_row.to_frame().T)

clipped = metrics_table.query("`Prediction Variant` == 'Clipped [0,100]'")
feature_effect = clipped.pivot(index="Model", columns="Experiment", values=["MAE","RMSE","R2"])
display(feature_effect)

time_bias = []
for model in MODEL_NAMES:
    with_time = clipped.query("Model == @model and Experiment == 'Engineered With-Time'").iloc[0]
    no_time = clipped.query("Model == @model and Experiment == 'Engineered No-Time'").iloc[0]
    time_bias.append({"Model":model, "With-Time RMSE":with_time.RMSE, "No-Time RMSE":no_time.RMSE,
                      "No-Time minus With-Time RMSE":no_time.RMSE-with_time.RMSE,
                      "Interpretation":"positive means time features improve held-out RMSE" if no_time.RMSE-with_time.RMSE > 0 else "non-positive means no-time is equal/better"})
time_bias_table = pd.DataFrame(time_bias)
display(time_bias_table)

FINAL WINNER: ('Baseline With-Time', 'XGBoost')


,Experiment,Model,Prediction Variant,Feature Count,MAE,RMSE,R2
0,Baseline With-Time,XGBoost,"Clipped [0,100]",18,16.302257,20.576374,0.260362


MAE                                          \
Experiment        Baseline With-Time Engineered No-Time Engineered With-Time   
Model                                                                          
Linear Regression          18.633607          21.672246            18.335150   
Random Forest              16.520659          19.812843            16.369137   
XGBoost                    16.302257          19.563436            16.201599   

                                RMSE                                          \
Experiment        Baseline With-Time Engineered No-Time Engineered With-Time   
Model                                                                          
Linear Regression          22.913068          24.733624            22.700191   
Random Forest              20.707796          23.179310            20.683873   
XGBoost                    20.576374          22.963482            20.594952   

                                  R2                                          
Experiment        Baseline With-Time Engineered No-Time Engineered With-Time  
Model                                                                         
Linear Regression           0.082834          -0.068703             0.099797  
Random Forest               0.250884           0.061396             0.252613  
XGBoost                     0.260362           0.078793             0.259026

,Model,With-Time RMSE,No-Time RMSE,No-Time minus With-Time RMSE,Interpretation
0,Linear Regression,22.700191,24.733624,2.033433,positive means time features improve held-out ...
1,Random Forest,20.683873,23.179310,2.495436,positive means time features improve held-out ...
2,XGBoost,20.594952,22.963482,2.368530,positive means time features improve held-out ...


## 4. Error diagnostics theo popularity group

Residual/Bias convention: **actual − prediction**. Bias dương = underprediction; bias âm = overprediction.

In [5]:
final_pred_raw = predictions[winner_key]
final_pred = np.clip(final_pred_raw, 0, 100)
diagnostics = pd.DataFrame({"Actual":np.asarray(y_test), "Prediction Raw":final_pred_raw, "Prediction Clipped":final_pred})
diagnostics["Residual (Actual-Prediction)"] = diagnostics["Actual"] - diagnostics["Prediction Clipped"]
diagnostics["Popularity Group"] = pd.cut(diagnostics["Actual"], [-np.inf,29,49,69,np.inf], labels=["Low 0-29","Emerging 30-49","Medium 50-69","High 70-100"])

def group_metrics(group):
    residual = group["Residual (Actual-Prediction)"].to_numpy()
    return pd.Series({"Rows":len(group), "MAE":np.abs(residual).mean(), "RMSE":np.sqrt(np.mean(residual**2)), "Bias (Actual-Prediction)":residual.mean(), "Bias Direction":"underprediction" if residual.mean()>0 else "overprediction"})
error_groups = diagnostics.groupby("Popularity Group", observed=True).apply(group_metrics, include_groups=False).reset_index()
display(error_groups)
display(pd.DataFrame(regression_metric_variants(y_test, final_pred_raw)).T)

,Popularity Group,Rows,MAE,RMSE,Bias (Actual-Prediction),Bias Direction
0,Low 0-29,8155,26.724760,30.514793,-25.242515,overprediction
1,Emerging 30-49,7760,7.621582,10.036774,-1.500985,overprediction
2,Medium 50-69,13122,12.271603,14.333525,12.098521,underprediction
3,High 70-100,3088,27.719605,28.624173,27.719605,underprediction


,MAE,RMSE,R2
Raw,16.313208,20.579572,0.260132
"Clipped [0,100]",16.302257,20.576374,0.260362


## 5. Feature importance, grouped importance và artifacts

In [6]:
detailed_importance = transformed_feature_importance(final_pipeline)
grouped_importance = grouped_feature_importance(final_pipeline)
display(detailed_importance.head(25))
display(grouped_importance.head(25))

metrics_table.to_csv(EVAL_DIR / "hotfix_all_experiment_metrics.csv", index=False)
time_bias_table.to_csv(EVAL_DIR / "hotfix_time_bias_comparison.csv", index=False)
error_groups.to_csv(EVAL_DIR / "hotfix_error_groups.csv", index=False)
detailed_importance.to_csv(EVAL_DIR / "hotfix_transformed_feature_importance.csv", index=False)
grouped_importance.to_csv(EVAL_DIR / "hotfix_grouped_feature_importance.csv", index=False)
pd.DataFrame({"track_id":raw.loc[test_mask, "track_id"].astype(str).values,
              "actual":np.asarray(y_test), "prediction_raw":final_pred_raw,
              "prediction_clipped":final_pred}).to_parquet(EVAL_DIR / "hard_requirement_test_predictions.parquet", index=False)

final_model_path = MODEL_DIR / "popularity_pipeline.joblib"
joblib.dump(final_pipeline, final_model_path, compress=3)
final_contract_features = get_model_features(include_engineered=winner_experiment["include_engineered"], include_time=winner_experiment["include_time"])
metrics_payload = {
    "final_experiment":winner_key[0], "final_model":winner_key[1],
    "include_engineered":winner_experiment["include_engineered"],
    "include_time":winner_experiment["include_time"],
    "model_features":final_contract_features,
    "selected_engineered_features":SELECTED_ENGINEERED_FEATURES,
    "selection_pool":"all eligible experiments and all three algorithms",
    "selection_metric":"minimum clipped [0,100] RMSE; MAE tie-break",
    "final_test_metrics":{"MAE":float(winner_row.MAE), "RMSE":float(winner_row.RMSE), "R2":float(winner_row.R2)},
    "raw_test_metrics":regression_metric_variants(y_test, final_pred_raw)["Raw"],
    "residual_convention":"actual - prediction; positive means underprediction",
    "parity_test":parity_result,
}
(MODEL_DIR / "metrics.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
(MODEL_DIR / "feature_columns.json").write_text(json.dumps(final_contract_features, indent=2), encoding="utf-8")
(EVAL_DIR / "model_metrics.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

reloaded_pipeline = joblib.load(final_model_path)
assert np.allclose(reloaded_pipeline.predict(X_test.head(20)), final_pipeline.predict(X_test.head(20)))
print(json.dumps(metrics_payload, indent=2))
print("Saved and reload-validated:", final_model_path)

,Feature,Importance
0,decade_2000,0.150902
1,release_year,0.109408
2,decade_1990,0.091784
3,decade_1920,0.088314
4,decade_2010,0.087622
5,decade_1950,0.072221
6,release_month_12.0,0.046420
7,decade_1960,0.045199
8,decade_1980,0.032267
9,decade_1970,0.030548


,Feature Group,Importance
0,decade,0.631629
1,release_year,0.109408
2,release_month,0.085758
3,key,0.028688
4,acousticness,0.027167
5,explicit,0.023850
6,release_precision,0.018942
7,instrumentalness,0.018220
8,loudness,0.009300
9,duration_min,0.008334


{
  "final_experiment": "Baseline With-Time",
  "final_model": "XGBoost",
  "include_engineered": false,
  "include_time": true,
  "model_features": [
    "duration_min",
    "release_year",
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "explicit",
    "mode",
    "release_month",
    "decade",
    "release_precision",
    "key",
    "time_signature"
  ],
  "selected_engineered_features": [
    "key_sin",
    "key_cos",
    "dance_energy",
    "positive_energy",
    "acoustic_energy_balance",
    "dance_valence",
    "acoustic_instrumental",
    "tempo_energy",
    "energy_vs_period_avg",
    "dance_vs_period_avg",
    "energy_loudness",
    "mood_quadrant",
    "duration_category",
    "tempo_category"
  ],
  "selection_pool": "all eligible experiments and all three algorithms",
  "selection_metric": "minimum clipped [0,100] RMSE; MAE tie-break",
  "final_test_metrics": {

## 6. Kết luận

Winner bên trên là kết quả đo mới sau hotfix, không tái sử dụng metric cũ. Bảng time-bias định lượng mức phụ thuộc vào release-time; bảng group errors nêu rõ khu vực model under/overpredict. Deployment phải đọc `include_engineered`, `include_time` và `model_features` từ chính artifact metadata này.